In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Personal Knowledge Base — RAG Over Your Own Notes

## 1. Project Overview

**Task:** Index a collection of personal notes, markdown files, and text files, then answer questions over them using retrieval-augmented generation (RAG). Explain every stage: ingestion, chunking, embeddings, retrieval, and answer generation.

**Approach:** Load files → parse markdown/text → chunk with metadata → embed locally → store in ChromaDB → retrieve → generate grounded answers with citations.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting, orchestration

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Ingest** heterogeneous files — `.md`, `.txt`, plain notes |
| 2 | **Parse markdown** — extract headings, sections, and metadata |
| 3 | **Chunk text** — understand chunk size, overlap, and their trade-offs |
| 4 | **Embed text** — convert chunks to dense vectors with sentence-transformers |
| 5 | **Store & retrieve** — build a vector store and perform similarity search |
| 6 | **Generate answers** — ground LLM responses in retrieved context |
| 7 | **Evaluate quality** — measure retrieval accuracy and answer faithfulness |

## 3. Problem Statement

We all accumulate notes — meeting notes, learning journals, project docs, book highlights, idea scratchpads. Over months it becomes impossible to remember where something was written. Standard file search (Ctrl+F) only finds exact words; it can't answer *questions*.

A personal knowledge base with RAG lets you ask:
- *"What were the key decisions from last month's design review?"*
- *"What did I learn about transformers vs RNNs?"*
- *"What books have I read about systems design?"*

## 4. Why This Project Matters

- **End-to-end RAG** — walks through every stage from raw files to grounded answers
- **Practical value** — build a tool you'll actually use on your own notes
- **Deep explanations** — each stage (ingest, chunk, embed, retrieve, generate) is explained with diagrams and experiments
- **Local & private** — your notes never leave your machine

## 5. Pipeline Architecture

```
Your Notes (markdown, txt, plain text)
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 1: INGESTION                      │
  │  • Scan folder for .md / .txt files      │
  │  • Extract text + file metadata           │
  │  • Parse markdown headings as sections    │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 2: CHUNKING                       │
  │  • Split into 400-char chunks            │
  │  • 80-char overlap for context continuity │
  │  • Preserve file name + heading metadata  │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 3: EMBEDDING                      │
  │  • all-MiniLM-L6-v2 (384-dim vectors)   │
  │  • Each chunk → one dense vector         │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 4: STORAGE (ChromaDB)             │
  │  • Vectors + text + metadata indexed     │
  │  • Cosine similarity for retrieval       │
  └──────────────────────────────────────────┘
         │
   User Question
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 5: RETRIEVAL                      │
  │  • Embed the question                    │
  │  • Find top-k most similar chunks        │
  │  • Optional metadata filter              │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 6: GENERATION                     │
  │  • Pack retrieved chunks as context      │
  │  • LLM generates grounded answer         │
  │  • Citations reference source files       │
  └──────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from pathlib import Path
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("Configuration")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Chunk: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval: top-{TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Ingestion

## 10. What Is Ingestion?

**Ingestion** is the first stage of any RAG pipeline. It means:

1. **Discover** files — scan a folder for `.md`, `.txt`, or other text files
2. **Load** content — read each file as a string
3. **Extract metadata** — file name, path, type, creation date, detected headings
4. **Normalize** — strip non-content elements (HTML tags, excessive whitespace)

### Why Metadata Matters

Without metadata, all chunks look the same after embedding. You can't:
- Search "only in my meeting notes" (file-based filter)
- Cite the source file in an answer
- Prioritize recent notes over old ones

### Supported Formats

| Format | Extensions | Parsing Strategy |
|--------|-----------|------------------|
| Markdown | `.md` | Extract `#` headings as section markers |
| Plain text | `.txt` | Treat as one section, use filename as title |
| Structured notes | `.md` with YAML front matter | Parse tags, date, title from front matter |

## 11. Sample Notes Collection

We create a realistic set of personal notes in-memory. In production, you'd point to your actual notes folder.

In [ ]:
SAMPLE_NOTES = {
    "meeting_notes_2026-03-15.md": """# Design Review — Search Service Rewrite

## Attendees
Alice (tech lead), Bob (backend), Carol (ML), Dave (PM)

## Key Decisions
- **Switch from Elasticsearch to Typesense** for the product search service. Reasons:
  faster cold-start, simpler ops, built-in typo tolerance.
- Keep Elasticsearch for log analytics — different use case, already stable.
- Target migration date: April 30. Bob owns the migration plan.
- ML ranking model (Carol) will be integrated in phase 2 (June).

## Action Items
- Bob: Write migration plan by March 22.
- Carol: Benchmark Typesense relevance vs Elasticsearch on our test queries.
- Dave: Update the product roadmap to reflect the new timeline.
- Alice: Schedule a follow-up review for April 5.

## Open Questions
- Do we need to maintain backward compatibility with the v1 search API?
- What's the fallback plan if Typesense can't handle our 50M document index?""",

    "learning_transformers.md": """# Transformers — Study Notes

## What Are Transformers?
Transformers are a neural network architecture introduced in "Attention Is All You
Need" (Vaswani et al., 2017). They replaced RNNs and LSTMs for most NLP tasks
because they can process all tokens in parallel (not sequentially).

## Self-Attention Mechanism
Self-attention computes a weighted sum of all positions in the input sequence for
each position. The weights are learned through Query (Q), Key (K), and Value (V)
matrices. Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V.

The sqrt(d_k) scaling prevents the dot products from growing too large, which would
push softmax into regions with tiny gradients.

## Multi-Head Attention
Instead of one attention function, transformers use multiple "heads" that attend to
different parts of the representation. Each head has its own Q, K, V projections.
Outputs are concatenated and linearly projected.

## Transformers vs RNNs
| Feature | RNN/LSTM | Transformer |
|---------|----------|-------------|
| Processing | Sequential | Parallel |
| Long-range dependencies | Struggles (vanishing gradients) | Handles well (self-attention) |
| Training speed | Slow (can't parallelize time steps) | Fast (all positions at once) |
| Memory | O(1) per step | O(n^2) attention matrix |

## Key Insight
Transformers trade compute for quality — the O(n^2) attention cost is worth it
because they learn much better representations of language.""",

    "book_notes_systems_design.txt": """Book: Designing Data-Intensive Applications by Martin Kleppmann

Chapter 1 — Reliable, Scalable, Maintainable Applications
- Reliability: system works correctly even when faults occur. Hardware faults,
  software errors, human errors. Use redundancy, testing, monitoring.
- Scalability: ability to handle increased load. Describe load with load parameters
  (e.g., requests/sec, read/write ratio). Measure performance: throughput, latency
  (p50, p99). Scaling up (bigger machine) vs scaling out (more machines).
- Maintainability: make it easy to work with over time. Operability (easy to
  operate), simplicity (manage complexity), evolvability (easy to change).

Chapter 3 — Storage and Retrieval
- Two main families of storage engines: log-structured (LSM trees, SSTables) and
  page-oriented (B-trees).
- LSM trees: write-optimized. Append-only. Compaction merges segments. Used by
  LevelDB, RocksDB, Cassandra.
- B-trees: read-optimized. In-place updates. Used by most relational databases.
- OLTP vs OLAP: transaction processing (many small reads/writes) vs analytics
  (few large read-only queries over many rows).

Chapter 5 — Replication
- Why replicate: keep data close to users (reduce latency), allow system to continue
  working if some parts fail, scale out read throughput.
- Leader-follower replication: one node accepts writes (leader), followers replicate.
  Synchronous vs asynchronous replication trade-offs.
- Eventual consistency: if you stop writing, all replicas eventually converge.
  Read-after-write consistency is stronger.""",

    "project_ideas.md": """# Project Ideas Backlog

## Active Ideas
- **Personal Knowledge Base**: Index all my notes and enable semantic search with
  RAG. Use ChromaDB + local embeddings. Privacy-first — no cloud APIs.
- **Expense Tracker CLI**: Simple terminal app to log expenses, categorize them
  automatically (LLM-based), and generate monthly reports as markdown.
- **Git Commit Summarizer**: Tool that reads a week of git commits and generates
  a human-readable weekly summary for standup reports.

## Parked Ideas
- Podcast transcription + summarization pipeline (need to test Whisper perf first)
- Recipe recommendation engine based on pantry ingredients
- Home automation dashboard with sensor data visualization

## Criteria for Next Project
1. Must be personally useful (I'll actually use it)
2. Can be built in < 2 weekends
3. Teaches me something new (new library, technique, or domain)
4. Runs locally — no recurring cloud costs""",

    "weekly_log_2026-03-17.md": """# Weekly Log — March 17-21, 2026

## Monday
- Started reading about vector databases. Pinecone vs Chroma vs Weaviate.
  ChromaDB seems best for local/offline use. Pinecone is cloud-only.
- Set up Ollama with qwen3 model. Works great for summarization tasks.

## Tuesday
- Pair programmed with Alice on the search service migration plan.
- Discovered that Typesense supports geosearch — useful for our location feature.
- Read 2 chapters of DDIA (Chapters 3 and 5).

## Wednesday
- Benchmark results: Typesense is 3x faster than Elasticsearch for our query
  patterns but relevance is slightly worse on fuzzy queries. Carol will tune.
- Wrote first draft of the migration plan. Bob reviewed and approved.

## Thursday
- Experimented with sentence-transformers for embedding quality.
  all-MiniLM-L6-v2 is good enough for 90% of cases. all-mpnet-base-v2 is
  better but 3x slower.
- Team lunch — discussed hackathon ideas for Q2.

## Friday
- Finished the migration plan doc and shared with stakeholders.
- Started prototyping the personal knowledge base project.
- Weekend goal: finish the chunking + embedding pipeline.""",

    "python_tips.txt": """Python Tips and Patterns I Keep Forgetting

--- Collections ---
Counter: from collections import Counter; Counter(['a','b','a']) => Counter({'a':2,'b':1})
defaultdict: from collections import defaultdict; d = defaultdict(list); d['key'].append(1)
namedtuple: from collections import namedtuple; Point = namedtuple('Point', 'x y')

--- Itertools ---
chain: itertools.chain([1,2], [3,4]) => 1,2,3,4 (flatten one level)
groupby: must sort first! itertools.groupby(sorted(data), key=func)
product: cartesian product. itertools.product('AB', '12') => A1, A2, B1, B2

--- Pathlib ---
Path.home() / 'Documents' / 'notes.txt'
path.stem => filename without extension
path.suffix => '.txt'
path.read_text() => reads entire file as string
list(Path('.').glob('**/*.md')) => find all markdown files recursively

--- F-strings ---
f"{value:,.2f}" => 1,234.56 (comma separator, 2 decimals)
f"{name!r}" => repr() of name (useful for debugging)
f"{dt:%Y-%m-%d}" => date formatting inside f-string

--- Context Managers ---
from contextlib import contextmanager
@contextmanager
def timer():
    start = time.time()
    yield
    print(f"Elapsed: {time.time() - start:.2f}s")""",
}

print(f"Sample knowledge base: {len(SAMPLE_NOTES)} files")
for name, content in SAMPLE_NOTES.items():
    ext = Path(name).suffix
    print(f"  {ext:>4} | {name}: {len(content):,} chars, ~{len(content.split()):,} words")

## 12. File Parsing and Metadata Extraction

In [ ]:
def parse_note(filename: str, content: str) -> list[dict]:
    """
    Parse a note file into sections with metadata.
    For markdown: split on ## headings.
    For txt: treat as one section.
    Returns list of {text, metadata}.
    """
    ext = Path(filename).suffix.lower()
    stem = Path(filename).stem
    file_type = "markdown" if ext == ".md" else "text"

    # Detect date from filename (common pattern: *_YYYY-MM-DD*)
    date_match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
    file_date = date_match.group(1) if date_match else "unknown"

    sections = []

    if ext == ".md":
        # Split on markdown headings (## or #)
        # Keep heading with its content
        parts = re.split(r"(?=^#{1,3}\s)", content, flags=re.MULTILINE)
        for part in parts:
            part = part.strip()
            if not part:
                continue
            # Extract heading
            heading_match = re.match(r"^#{1,3}\s+(.+)", part)
            heading = heading_match.group(1).strip() if heading_match else stem
            sections.append({
                "text": part,
                "metadata": {
                    "filename": filename,
                    "file_type": file_type,
                    "heading": heading,
                    "date": file_date,
                },
            })
    else:
        # Plain text — one section per file
        # Try to extract a title from the first line
        first_line = content.split("\n")[0].strip()
        sections.append({
            "text": content,
            "metadata": {
                "filename": filename,
                "file_type": file_type,
                "heading": first_line[:60],
                "date": file_date,
            },
        })

    return sections


# Parse all notes
all_sections = []
for filename, content in SAMPLE_NOTES.items():
    secs = parse_note(filename, content)
    all_sections.extend(secs)

print(f"Parsed {len(all_sections)} sections from {len(SAMPLE_NOTES)} files")
for s in all_sections[:6]:
    m = s["metadata"]
    print(f"  [{m['file_type']:>8}] {m['filename'][:35]:35s} → {m['heading'][:40]}")

---

# Part B — Chunking

## 13. What Is Chunking and Why Does It Matter?

### The Problem
Embedding models have a **fixed input size** (typically 256-512 tokens). Long documents must be split into smaller pieces. But *how* you split matters enormously:

| Chunk Size | Pros | Cons |
|------------|------|------|
| **Too small** (< 100 chars) | Very precise retrieval | Loses context, fragments sentences |
| **Too large** (> 1000 chars) | Full context in each chunk | Embeds too many topics per chunk; diluted similarity |
| **Sweet spot** (300-600 chars) | Good balance of context and precision | Requires overlap to avoid cutting sentences |

### Overlap
Overlap means each chunk shares some text with its neighbors. This prevents losing information at chunk boundaries:

```
Chunk 1: [------- 400 chars -------]
Chunk 2:            [------- 400 chars -------]
                ^^^^  80-char overlap
```

### Splitting Strategy
`RecursiveCharacterTextSplitter` tries to split on natural boundaries in order of preference:
1. `\n\n` (paragraph break) — best, preserves paragraphs
2. `\n` (line break) — good, preserves lines
3. `. ` (sentence end) — acceptable, preserves sentences
4. ` ` (word boundary) — last resort, never splits mid-word

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", ", ", " "],
)

all_chunks = []

for section in all_sections:
    text = section["text"]
    metadata = section["metadata"]
    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "metadata": {
                **metadata,
                "chunk_index": i,
                "chunk_total": len(chunks),
            },
        })

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} chars")

# Breakdown by file
from collections import Counter
file_counts = Counter(c["metadata"]["filename"] for c in all_chunks)
print(f"\nChunks per file:")
for fname, count in file_counts.items():
    print(f"  {fname}: {count} chunks")

# Show sample chunk with metadata
s = all_chunks[3]
print(f"\nSample chunk:")
print(f"  File: {s['metadata']['filename']}")
print(f"  Heading: {s['metadata']['heading']}")
print(f"  Chunk {s['metadata']['chunk_index']+1}/{s['metadata']['chunk_total']}")
print(f"  Text: {s['text'][:150]}...")

---

# Part C — Embeddings

## 14. What Are Embeddings?

An **embedding** is a dense vector (list of numbers) that captures the *meaning* of a text.

### How It Works
```
"sick leave policy"  →  [0.12, -0.34, 0.56, ..., 0.78]  (384 dimensions)
"vacation days off"  →  [0.11, -0.33, 0.55, ..., 0.77]  (very similar vector!)
"neural network arch" → [0.89, 0.12, -0.67, ..., -0.34]  (very different vector)
```

Texts with similar meaning → similar vectors → small cosine distance.

### Why `all-MiniLM-L6-v2`?

| Model | Dimensions | Speed | Quality | Size |
|-------|-----------|-------|---------|------|
| `all-MiniLM-L6-v2` | 384 | ⚡ Fast | Good (90%) | 80 MB |
| `all-mpnet-base-v2` | 768 | Medium | Best (95%) | 420 MB |
| `all-MiniLM-L12-v2` | 384 | Medium | Better (92%) | 120 MB |

For a personal knowledge base, `all-MiniLM-L6-v2` is the right trade-off: fast enough to re-index on every query, small enough to run on any machine.

In [ ]:
# Initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")

# Demonstrate embedding
sample_texts = [
    "How does self-attention work in transformers?",
    "The attention mechanism computes query-key-value matrices.",
    "What are the best Python libraries for data analysis?",
]

sample_embeds = embeddings.embed_documents(sample_texts)

print(f"\nEmbedding dimension: {len(sample_embeds[0])}")
print(f"\nSample vector (first 10 values):")
for text, emb in zip(sample_texts, sample_embeds):
    vec_preview = ", ".join(f"{v:.3f}" for v in emb[:10])
    print(f"  '{text[:45]}...'")
    print(f"    [{vec_preview}, ...]")

# Show cosine similarity between pairs
import numpy as np
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nCosine similarities:")
print(f"  Text 1 vs Text 2 (both about attention): {cosine_sim(sample_embeds[0], sample_embeds[1]):.3f}")
print(f"  Text 1 vs Text 3 (different topics):     {cosine_sim(sample_embeds[0], sample_embeds[2]):.3f}")
print(f"  → Higher similarity = more semantically related")

## 15. Build the Vector Store

In [ ]:
# Initialize ChromaDB (in-memory for this notebook)
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("personal_kb")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="personal_kb",
    metadata={"hnsw:space": "cosine"},
)

# Embed all chunks
texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

print("Embedding all chunks...")
all_embeds = []
batch_size = 32
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    all_embeds.extend(embeddings.embed_documents(batch))
    print(f"  Batch {i // batch_size + 1}/{(len(texts) - 1) // batch_size + 1} done")

collection.add(
    documents=texts,
    embeddings=all_embeds,
    metadatas=metadatas,
    ids=ids,
)

print(f"\n✅ Vector store built: {collection.count()} chunks indexed")

---

# Part D — Retrieval

## 16. What Is Retrieval?

**Retrieval** finds the most relevant chunks for a user's question:

1. **Embed the question** using the same embedding model
2. **Compare** against all stored chunk vectors using cosine similarity
3. **Return top-k** most similar chunks with their metadata

### Cosine Similarity
$$\text{similarity}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \cdot \|\vec{b}\|}$$

- `1.0` = identical meaning
- `0.0` = unrelated
- Values above `0.6` are usually relevant for notes retrieval

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             where: Optional[dict] = None) -> list[dict]:
    """Retrieve top-k chunks for a query, with optional metadata filter."""
    query_embed = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_embed], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def display_chunks(chunks: list[dict]):
    """Pretty print retrieved chunks."""
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c["distance"]
        print(f"\n  [{i}] (sim={sim:.3f}) {m['filename']} → {m['heading'][:40]}")
        print(f"      {c['text'][:130]}...")


# Test retrieval
q = "What were the key decisions from the design review?"
chunks = retrieve(q)

print(f"Query: {q}")
print(f"Retrieved {len(chunks)} chunks:")
display_chunks(chunks)

## 17. Retrieval With Metadata Filters

In [ ]:
print("FILTERED RETRIEVAL EXAMPLES")
print("=" * 60)

# Filter by file type
q = "What Python patterns should I remember?"
print(f"\nQ: {q}")
print(f"Filter: file_type = 'text'")
chunks_txt = retrieve(q, where={"file_type": "text"})
display_chunks(chunks_txt[:3])

# Filter by date
q = "What did I work on last week?"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print(f"Filter: date = '2026-03-17'")
chunks_dated = retrieve(q, where={"date": "2026-03-17"})
display_chunks(chunks_dated[:3])

# Filter by filename
q = "What books have I read?"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print(f"No filter (all files):")
chunks_all = retrieve(q, top_k=3)
display_chunks(chunks_all)

---

# Part E — Answer Generation

## 18. What Is the Generation Stage?

Generation is where the LLM reads the retrieved chunks and produces a natural language answer:

```
┌──────────────────────────────┐
│  System Prompt               │
│  "You are a personal         │
│   knowledge assistant..."    │
├──────────────────────────────┤
│  User Prompt                 │
│  Question: ...               │
│  Source 1: [file, heading]   │
│    chunk text...             │
│  Source 2: [file, heading]   │
│    chunk text...             │
├──────────────────────────────┤
│  LLM Output                  │
│  Grounded answer with        │
│  [Source: file] citations    │
└──────────────────────────────┘
```

### Key Rules for Grounded Generation
1. **Only use information from provided sources** — no outside knowledge
2. **Cite sources** — so the user can verify
3. **Say "I don't know"** when sources don't cover the question
4. **Use low temperature** (0.1) — less creative, more faithful

In [ ]:
RAG_SYSTEM = """You are a personal knowledge assistant. You answer questions using
ONLY the user's own notes provided as source documents.

Rules:
1. Base your answer ONLY on the provided sources. Never use outside knowledge.
2. Cite sources as [Source: filename → heading].
3. If the notes don't contain the answer, say: "I couldn't find this in your notes."
4. Be specific — include names, dates, numbers from the notes.
5. Keep answers concise — 2-5 sentences unless more detail is needed."""

RAG_PROMPT = """Answer this question using my notes.

QUESTION: {question}

MY NOTES (sources):
{sources}

Return ONLY JSON:
{{
  "answer": "grounded answer with [Source: filename] citations",
  "confidence": "high|medium|low",
  "confidence_reason": "why this confidence level",
  "sources_used": ["filename — heading"]
}}"""


def format_sources(chunks: list[dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        parts.append(
            f"[Source {i}: {m['filename']} → {m['heading']}]\n{c['text']}\n"
        )
    return "\n".join(parts)


def ask_kb(question: str, where: Optional[dict] = None) -> dict:
    """Full RAG pipeline: retrieve → format → generate → parse."""
    chunks = retrieve(question, where=where)
    sources_text = format_sources(chunks)

    resp = ask(
        RAG_PROMPT.format(question=question, sources=sources_text),
        system=RAG_SYSTEM,
        temperature=TEMP_ANSWER,
    )
    result = parse_json(resp)
    if result:
        result["chunks"] = chunks
    else:
        result = {"answer": resp, "confidence": "unknown", "chunks": chunks}
    return result


print("ask_kb() pipeline defined")

## 19. Ask Questions Over Your Notes

In [ ]:
def display_answer(question: str, result: dict):
    """Pretty-print a Q&A result with sources."""
    conf = result.get("confidence", "?")
    badge = {"high": "🟢 HIGH", "medium": "🟡 MEDIUM", "low": "🔴 LOW"}.get(conf, "⚪ ?")

    print(f"\n{'═'*70}")
    print(f"  Q: {question}")
    print(f"  Confidence: {badge}")
    print(f"{'═'*70}")
    print()
    wrap_print(result.get("answer", ""))

    print(f"\n  {'─'*66}")
    print(f"  Sources:")
    for s in result.get("sources_used", []):
        print(f"    📄 {s}")

    # Show top 3 retrieved chunks
    chunks = result.get("chunks", [])[:3]
    print(f"\n  Retrieved chunks ({len(chunks)} shown):")
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c.get("distance", 1)
        print(f"    [{i}] {m['filename']} → {m['heading'][:35]} (sim={sim:.3f})")


# Question 1: Factual recall from meeting notes
q1 = "What search engine are we migrating to and who owns the migration plan?"
r1 = ask_kb(q1)
display_answer(q1, r1)

In [ ]:
# Question 2: Learning notes
q2 = "What is self-attention and how does it work?"
r2 = ask_kb(q2)
display_answer(q2, r2)

In [ ]:
# Question 3: Cross-file synthesis
q3 = "What embedding models did I evaluate and which did I pick?"
r3 = ask_kb(q3)
display_answer(q3, r3)

In [ ]:
# Question 4: Book notes
q4 = "What did I learn about replication and consistency from DDIA?"
r4 = ask_kb(q4)
display_answer(q4, r4)

In [ ]:
# Question 5: Out of scope — test confidence
q5 = "What is my favorite restaurant and what should I order there?"
r5 = ask_kb(q5)
display_answer(q5, r5)

conf = r5.get("confidence", "")
print(f"\n  → {'✅' if conf == 'low' else '⚠'} Expected LOW confidence for out-of-scope question")

## 20. Batch Q&A Over the Knowledge Base

In [ ]:
batch_questions = [
    "What project ideas do I have that run locally?",
    "How do transformers compare to RNNs?",
    "What happened with Typesense benchmarking?",
    "What is the itertools groupby gotcha?",
    "What action items came out of the design review?",
    "What chapters of DDIA have I read?",
]

print("BATCH Q&A")
print("=" * 60)

for q in batch_questions:
    result = ask_kb(q)
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")
    answer = result.get("answer", "")
    print(f"\n  {badge} Q: {q}")
    print(f"     A: {textwrap.shorten(answer, width=100, placeholder='...')}")

---

# Part F — Evaluation & Experiments

## 21. Retrieval Accuracy Test

In [ ]:
retrieval_tests = [
    {"query": "What is the migration timeline for the search service?",
     "expected_file": "meeting_notes_2026-03-15.md"},
    {"query": "How does multi-head attention work?",
     "expected_file": "learning_transformers.md"},
    {"query": "What does Kleppmann say about storage engines?",
     "expected_file": "book_notes_systems_design.txt"},
    {"query": "What are my criteria for choosing the next project?",
     "expected_file": "project_ideas.md"},
    {"query": "What did I do on Wednesday this week?",
     "expected_file": "weekly_log_2026-03-17.md"},
    {"query": "How do I use Path in Python?",
     "expected_file": "python_tips.txt"},
]

print("RETRIEVAL ACCURACY")
print("=" * 60)

correct = 0
for test in retrieval_tests:
    chunks = retrieve(test["query"], top_k=3)
    top_file = chunks[0]["metadata"]["filename"]
    match = top_file == test["expected_file"]
    if match:
        correct += 1
    status = "✅" if match else "❌"
    print(f"\n  {status} Q: {test['query'][:55]}")
    print(f"      Expected: {test['expected_file']}")
    print(f"      Got:      {top_file}")

print(f"\n{'─'*60}")
print(f"Accuracy: {correct}/{len(retrieval_tests)} ({correct/len(retrieval_tests)*100:.0f}%)")

## 22. Answer Faithfulness Evaluation

In [ ]:
EVAL_PROMPT = """Evaluate this RAG answer for faithfulness and quality.

Question: {question}
Answer: {answer}
Sources used:
{sources}

Score 1-5 on:
- FAITHFULNESS: Answer uses only info from sources (5=fully grounded)
- RELEVANCE: Answer addresses the question (5=perfectly relevant)
- COMPLETENESS: Answer covers all relevant info from sources (5=nothing missed)

Return ONLY JSON:
{{"faithfulness": N, "relevance": N, "completeness": N,
  "feedback": "one sentence"}}"""

eval_qs = [
    "What search engine are we migrating to?",
    "How does self-attention scale with sequence length?",
    "What are my active project ideas?",
]

print("ANSWER FAITHFULNESS EVALUATION")
print("=" * 60)

for q in eval_qs:
    result = ask_kb(q)
    sources_text = "\n".join(
        f"- [{c['metadata']['filename']}] {c['text'][:100]}..."
        for c in result.get("chunks", [])[:3]
    )
    eval_resp = ask(
        EVAL_PROMPT.format(
            question=q,
            answer=result.get("answer", "")[:400],
            sources=sources_text,
        ),
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(eval_resp)
    if scores:
        avg = sum(scores.get(k, 0) for k in ["faithfulness", "relevance", "completeness"]) / 3
        print(f"\n  Q: {q[:55]}")
        print(f"    Faith: {scores.get('faithfulness',0)}/5  "
              f"Rel: {scores.get('relevance',0)}/5  "
              f"Comp: {scores.get('completeness',0)}/5  "
              f"→ Avg: {avg:.1f}/5")
        print(f"    {scores.get('feedback', '')}")

## 23. Chunk Size Experiment

In [ ]:
def test_chunk_size(chunk_size: int, query: str, expected_file: str) -> dict:
    """Test retrieval accuracy at different chunk sizes."""
    sp = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_size // 5,
        separators=["\n\n", "\n", ". ", " "],
    )
    test_chunks = []
    for sec in all_sections:
        for i, ch in enumerate(sp.split_text(sec["text"])):
            test_chunks.append({"text": ch, "metadata": {**sec["metadata"], "chunk_index": i}})

    coll_name = f"cs_{chunk_size}"
    try:
        chroma_client.delete_collection(coll_name)
    except Exception:
        pass
    coll = chroma_client.create_collection(coll_name, metadata={"hnsw:space": "cosine"})

    t = [c["text"] for c in test_chunks]
    e = embeddings.embed_documents(t)
    coll.add(documents=t, embeddings=e,
             metadatas=[c["metadata"] for c in test_chunks],
             ids=[f"c{i}" for i in range(len(test_chunks))])

    qe = embeddings.embed_query(query)
    res = coll.query(query_embeddings=[qe], n_results=3)
    top_file = res["metadatas"][0][0]["filename"]
    top_sim = 1 - res["distances"][0][0]

    chroma_client.delete_collection(coll_name)

    return {"chunk_size": chunk_size, "num_chunks": len(test_chunks),
            "top_file": top_file, "top_sim": top_sim,
            "correct": top_file == expected_file}


test_q = "What were the benchmark results for Typesense vs Elasticsearch?"
expected = "weekly_log_2026-03-17.md"

print("CHUNK SIZE EXPERIMENT")
print("=" * 60)
print(f"Query: {test_q}")
print(f"Expected: {expected}\n")

for size in [150, 300, 400, 600, 1000]:
    r = test_chunk_size(size, test_q, expected)
    status = "✅" if r["correct"] else "❌"
    print(f"  {status} chunk={r['chunk_size']:>5} → {r['num_chunks']:>3} chunks | "
          f"top: {r['top_file'][:30]} (sim={r['top_sim']:.3f})")

## 24. Naive (No-RAG) vs RAG Comparison

In [ ]:
NAIVE_SYSTEM = """Answer the user's question based on your general knowledge.
Be honest if you don't know."""

q = "What were the key decisions from the design review meeting on March 15?"

# Naive (no context)
naive_answer = ask(q, system=NAIVE_SYSTEM, temperature=0.1)

# RAG (with context)
rag_result = ask_kb(q)
rag_answer = rag_result.get("answer", "")

print("NAIVE vs RAG COMPARISON")
print("=" * 60)
print(f"Q: {q}")

print(f"\n{'─'*60}")
print(f"❌ NAIVE (no notes):")
wrap_print(naive_answer)

print(f"\n{'─'*60}")
print(f"✅ RAG (with notes):")
wrap_print(rag_answer)

print(f"\n{'─'*60}")
print("→ RAG answers are grounded in YOUR notes — specific names, dates, decisions.")
print("→ Naive answers are generic or hallucinated because the LLM has no access to your data.")

---

## 25. Load Real Files (Production Pattern)

To use this on your own notes, replace the in-memory `SAMPLE_NOTES` with a folder scanner.

In [ ]:
def load_notes_from_folder(folder: str, extensions: tuple = (".md", ".txt")) -> dict:
    """
    Scan a folder recursively and load all matching text files.
    Returns {filename: content} dict.
    """
    notes = {}
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"Folder not found: {folder}")
        return notes

    for ext in extensions:
        for filepath in folder_path.rglob(f"*{ext}"):
            try:
                content = filepath.read_text(encoding="utf-8", errors="replace")
                # Use relative path as the filename key
                rel_path = str(filepath.relative_to(folder_path))
                notes[rel_path] = content
            except Exception as e:
                print(f"  Skip {filepath}: {e}")

    print(f"Loaded {len(notes)} files from {folder}")
    for name in sorted(notes)[:10]:
        print(f"  {name}: {len(notes[name]):,} chars")
    if len(notes) > 10:
        print(f"  ... and {len(notes) - 10} more")

    return notes


# Usage (uncomment and set your notes folder):
# my_notes = load_notes_from_folder(r"C:\Users\you\Documents\notes")
# Then re-run the ingestion → chunking → embedding pipeline with my_notes.

print("load_notes_from_folder() defined")
print("Usage: notes = load_notes_from_folder(r'path/to/your/notes')")

## 26. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **No metadata on chunks** | Can't cite sources or filter by file/date | Always attach filename, heading, date to each chunk |
| **Chunk size too large** | Embedding captures too many topics → poor retrieval precision | 300-600 chars is optimal for notes |
| **No overlap** | Sentences split mid-word; context lost at boundaries | Use ~20% overlap (80 chars for 400-char chunks) |
| **Using LLM without retrieval** | LLM hallucinates answers about your personal notes | Always retrieve first, then generate from context |
| **High temperature for answers** | Creative outputs deviate from source material | Use 0.1 for grounded answers |
| **No confidence signal** | Users trust wrong answers | Ask LLM to self-assess or compute retrieval similarity threshold |
| **Embedding query differently from docs** | Mismatch degrades retrieval | Use the same embedding model for both |
| **Stuffing all chunks into prompt** | Exceeds context window; slow; irrelevant noise | top-k=3-5 is optimal; more dilutes relevance |

## 27. Production Improvement Ideas

1. **Persistent vector store** — use `chromadb.PersistentClient(path=...)` so you don't re-embed on every startup
2. **Incremental indexing** — only embed new/changed files since last run (compare file hashes)
3. **Hybrid search** — combine BM25 keyword search with semantic search for better recall
4. **Re-ranking** — use a cross-encoder to re-rank top-20 results down to top-5
5. **Conversational memory** — support follow-up questions ("What else was discussed?")
6. **Folder watcher** — auto-re-index when files change (using `watchdog` library)
7. **Tag-based filtering** — parse YAML front matter (tags, categories) and expose them as filters
8. **Obsidian integration** — read Obsidian vault structure, respect `[[wikilinks]]` as relationships

## 28. Exercises

### Exercise 1: Index Your Own Notes

In [ ]:
# ── Exercise 1: Use your own notes folder ──────────────
# 1. Point to your real notes folder
# 2. Load, parse, chunk, embed, index
# 3. Ask 5 questions that only your notes can answer

# NOTES_FOLDER = r"C:\Users\you\Documents\notes"  # or ~/notes
# my_notes = load_notes_from_folder(NOTES_FOLDER)
#
# all_sections = []
# for fname, content in my_notes.items():
#     all_sections.extend(parse_note(fname, content))
#
# # Then re-run chunks → embed → store → query

print("Exercise 1: Index your own notes and ask questions.")
print("The pipeline handles .md and .txt files automatically.")

### Exercise 2: Compare Embedding Models

In [ ]:
# ── Exercise 2: Evaluate different embedding models ──────

# Test all-MiniLM-L6-v2 vs all-mpnet-base-v2 on the same retrieval tests.
# Measure: retrieval accuracy, similarity scores, embedding speed.

MODELS_TO_TEST = ["all-MiniLM-L6-v2", "all-MiniLM-L12-v2"]

test_q = "How do transformers handle long-range dependencies?"
expected_file = "learning_transformers.md"

print("EMBEDDING MODEL COMPARISON")
print("=" * 60)
print(f"Query: {test_q}")
print(f"Expected: {expected_file}\n")

import time

for model_name in MODELS_TO_TEST:
    emb = HuggingFaceEmbeddings(model_name=model_name)
    test_texts = [c["text"] for c in all_chunks[:20]]  # Subset for speed

    start = time.time()
    test_embeds = emb.embed_documents(test_texts)
    elapsed = time.time() - start

    q_embed = emb.embed_query(test_q)
    # Compute similarity to all test chunks
    sims = [cosine_sim(q_embed, e) for e in test_embeds]
    best_idx = max(range(len(sims)), key=lambda i: sims[i])
    best_file = all_chunks[best_idx]["metadata"]["filename"]

    match = "✅" if best_file == expected_file else "❌"
    print(f"  {match} {model_name}")
    print(f"      Dim: {len(test_embeds[0])} | Time: {elapsed:.2f}s | "
          f"Top: {best_file} (sim={sims[best_idx]:.3f})")

### Exercise 3: Summarize All Notes

In [ ]:
# ── Exercise 3: Generate a "knowledge map" of your notes ──

MAP_PROMPT = """Given these file titles and headings from a personal knowledge base,
create a knowledge map — group them by topic and describe what each group covers.

Files and headings:
{file_list}

Return ONLY JSON:
{{
  "topics": [
    {{
      "name": "topic name",
      "files": ["filename1", "filename2"],
      "summary": "what this topic covers"
    }}
  ],
  "total_coverage": "one sentence about the overall knowledge base"
}}"""

file_list = "\n".join(
    f"- {s['metadata']['filename']} → {s['metadata']['heading']}"
    for s in all_sections[:20]
)

map_resp = ask(MAP_PROMPT.format(file_list=file_list), temperature=0.2)
kmap = parse_json(map_resp)

print("KNOWLEDGE MAP")
print("=" * 60)

if kmap:
    for topic in kmap.get("topics", []):
        print(f"\n  📂 {topic.get('name', '')}")
        print(f"     {topic.get('summary', '')}")
        for f in topic.get("files", []):
            print(f"       └ {f}")
    print(f"\n  📊 {kmap.get('total_coverage', '')}")
else:
    wrap_print(map_resp[:400])

### Mini Challenge

1. **Similarity threshold:** Instead of always returning top-k, set a minimum similarity threshold (e.g., 0.5). If no chunk exceeds it, return "Not found in notes" without calling the LLM. Compare the false-positive rate vs always answering.

2. **Multi-turn follow-ups:** Implement a `chat_kb()` function that keeps conversation history. Ask: "What is Typesense?" → follow-up: "When is the migration deadline?" The second question should use context from both the notes AND the first answer.

3. **Incremental indexing:** Hash each note file (SHA-256). On re-index, only re-embed files whose hash has changed. Measure the time savings for a 100-file notes vault.

4. **Tag extraction:** For each note file, use the LLM to auto-generate 3-5 topic tags. Store them as metadata. Enable tag-based browsing: "Show me all notes tagged #distributed-systems."

## 29. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nKnowledge base: {len(SAMPLE_NOTES)} files → {len(all_chunks)} chunks")
print(f"File types: .md (markdown), .txt (plain text)")

print(f"\nRAG stages covered:")
print(f"  ✅ Stage 1: Ingestion — load files, extract metadata")
print(f"  ✅ Stage 2: Chunking — 400-char chunks, 80-char overlap")
print(f"  ✅ Stage 3: Embedding — all-MiniLM-L6-v2, 384 dimensions")
print(f"  ✅ Stage 4: Storage — ChromaDB with cosine similarity")
print(f"  ✅ Stage 5: Retrieval — top-k search with metadata filters")
print(f"  ✅ Stage 6: Generation — grounded answers with citations")

print(f"\nExperiments completed:")
print(f"  ✅ Retrieval accuracy test (6 queries)")
print(f"  ✅ Answer faithfulness evaluation")
print(f"  ✅ Chunk size experiment (150-1000)")
print(f"  ✅ Naive vs RAG comparison")
print(f"  ✅ Embedding model comparison")
print(f"  ✅ Out-of-scope detection")
print(f"  ✅ Knowledge map generation")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Ingestion** converts raw files to structured text with metadata (filename, heading, date) |
| 2 | **Chunking** splits text into 300-600 char pieces with ~20% overlap to preserve context |
| 3 | **Embeddings** map text to dense vectors; semantically similar text → similar vectors |
| 4 | **Retrieval** finds the top-k chunks by cosine similarity to the question embedding |
| 5 | **Generation** uses the LLM to synthesize retrieved chunks into a grounded answer |
| 6 | **Metadata filters** let you scope search by file, date, or type — critical for personal notes |
| 7 | **Citations** make answers verifiable — users can check the original note |
| 8 | **Always compare RAG vs naive** — the gap shows why retrieval matters |
| 9 | **Chunk size is a tunable parameter** — experiment with your own data to find the sweet spot |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*